# 강화학습 실습

**Reinforcement Learning · RL**

행동에 따른 보상을 반복 경험하며 좋은 정책을 학습하는 방법.

소재 분야에서 이해하기: 합성 공정의 순서와 조건 선택을 보상 기준으로 학습한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Gymnasium 강화학습 문서](https://gymnasium.farama.org/)

## 1. 공정 순서 선택을 보상으로 학습

5단계 온도 스케줄을 고르는 문제를 표 기반 Q 학습으로 풉니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

temperatures = [650, 750, 850]        # 행동: 다음 단계 온도
stages = 5

def reward(schedule):
    """가상 보상: 중간에 한 번 고온을 거치고 마지막은 저온이어야 좋습니다."""
    schedule = np.array(schedule)
    score = 0.0
    score += 1.0 if (schedule == 850).sum() == 1 else -0.3 * abs((schedule == 850).sum() - 1)
    score += 0.8 if schedule[-1] == 650 else -0.2
    score -= 0.05 * np.abs(np.diff(schedule)).sum() / 100.0
    return score

print('예시 스케줄 %s -> 보상 %.2f' % ([750, 850, 750, 750, 650], reward([750, 850, 750, 750, 650])))

In [ ]:
def q_learning(episodes=4000, epsilon=0.25, alpha=0.3, seed=0):
    local = np.random.default_rng(seed)
    # 상태 = (진행 단계, 고온 사용 횟수 0/1/2+)
    q = np.zeros((stages + 1, 3, len(temperatures)))
    history = []
    for episode in range(episodes):
        schedule, high = [], 0
        for stage in range(stages):
            state = (stage, min(high, 2))
            if local.random() < epsilon:
                action = local.integers(0, len(temperatures))
            else:
                action = int(np.argmax(q[state]))
            schedule.append(temperatures[action])
            high += temperatures[action] == 850
            next_state = (stage + 1, min(high, 2))
            future = 0.0 if stage == stages - 1 else q[next_state].max()
            immediate = reward(schedule) if stage == stages - 1 else 0.0
            q[state][action] += alpha * (immediate + future - q[state][action])
        history.append(reward(schedule))
    return q, np.array(history)

q, history = q_learning()
window = 200
smoothed = np.convolve(history, np.ones(window) / window, mode='valid')
plt.plot(smoothed); plt.xlabel('episode'); plt.ylabel('reward (moving average)'); plt.show()

In [ ]:
schedule, high = [], 0
for stage in range(stages):
    action = int(np.argmax(q[(stage, min(high, 2))]))
    schedule.append(temperatures[action]); high += temperatures[action] == 850
print('학습된 스케줄 %s -> 보상 %.3f' % (schedule, reward(schedule)))
best_possible = max(reward(list(combo)) for combo in __import__('itertools').product(temperatures, repeat=stages))
print('전수 탐색 최적 보상 %.3f' % best_possible)
print('\n보상 설계가 곧 목표 설계입니다. 보상을 잘못 정의하면 원하지 않는 정책을 학습합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#reinforcement-learning)을 여세요.